# Targeted Dropout Ladder: Stage C Official 500K Run

This notebook runs the official 500K Stage C experiment after the verified Stage B gate.
It reads the persisted Stage B acceptance manifest and permits only the two recommended
configurations: embedding dropout at 0.005 and residual dropout at 0.01. Raw `K` samples,
source-row identity, runtime configs, shard logs, and compact strategy outputs are retained.

Every scoring shard is independently validated and resumable. After a fresh-runtime reconnect,
rerun Sections 1-5, then Sections 9, 8, 10, and 11; validated work is skipped.

## 0. Resource Assumptions

- Runtime: Google Colab with Python 3.12 and PyTorch 2.10.x or 2.11.x. The current runtime is preferred; past runtime 2026.04 is also supported.
- GPU: T4, L4, A100, and comparable CUDA accelerators are supported.
- Source checkpoints and official 500K artifacts already exist under the project Google Drive root.
- Local scratch: about 1.2 GB for the Stage C token copy, plus Python/package overhead.
- Drive: allow roughly 4 GB for this two-config Stage C run. The scorer in
  the pinned commit must use bounded shard index files; the preflight cell enforces this.
- Default production shard size is 24,992 rows, matching the prior successful 500K run.
  Each shard is independently validated and restartable after a disconnect.
- The measured Stage B throughput implies roughly 6-8 hours on a T4 for both Stage C configs.
  Faster GPUs reduce this materially. The benchmark cell prints a runtime-specific estimate.
- Keep the Colab tab open while a cell is launching jobs. Completed shard outputs remain on Drive.

## 1. Runtime and Drive

In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if not gpu:
    raise RuntimeError('No CUDA GPU is visible. Select a GPU runtime before continuing.')
print('python:', sys.version)
print('platform:', platform.platform())
print('gpu:', gpu)
print('local disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

In [ ]:
# PYTHON CELL
DRIVE = Path('/content/drive/MyDrive/color-filter-ablation')
DATA_DRIVE = DRIVE / 'data'
SOURCE_RESULTS_DRIVE = DRIVE / 'results'
MODELS_DRIVE = DRIVE / 'assets' / 'raw' / 'models'
if not DRIVE.exists():
    raise FileNotFoundError(f'Project Drive root not found: {DRIVE}')

print('project Drive root:', DRIVE)
print('Drive disk free GB:', round(shutil.disk_usage('/content/drive').free / 1e9, 2))

## 2. Clone, Pin, and Install

In [ ]:
# PYTHON CELL
import re

OLMO_REPO = 'https://github.com/myazdani/color-filter-olmo.git'
PRODUCER_SHA = 'f7c8efad718ce65f0681b1f67e6016c2b4f00f7a'
ANALYSIS_SHA = '1adafd63779821c4803eb90b958cf6640a30f6fd'
NOTEBOOK_REVISION = 'stage-c-v3-2026-07-15'
OLMO_DIR = Path('/content/color-filter-olmo')

for label, revision in [('producer', PRODUCER_SHA), ('analysis', ANALYSIS_SHA)]:
    if not re.fullmatch(r'[0-9a-f]{40}', revision):
        raise RuntimeError(f'{label} revision must be a full pushed commit SHA: {revision!r}')
if not (OLMO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', OLMO_REPO, str(OLMO_DIR)], check=True)
for revision in (PRODUCER_SHA, ANALYSIS_SHA):
    subprocess.run(['git', '-C', str(OLMO_DIR), 'fetch', 'origin', revision], check=True)
subprocess.run(['git', '-C', str(OLMO_DIR), 'checkout', '--detach', ANALYSIS_SHA], check=True)
checked_out = subprocess.run(
    ['git', '-C', str(OLMO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
if checked_out != ANALYSIS_SHA:
    raise RuntimeError(f'Analysis checkout mismatch: expected {ANALYSIS_SHA}, found {checked_out}')
changed = subprocess.run(
    ['git', '-C', str(OLMO_DIR), 'diff', '--name-only', PRODUCER_SHA, ANALYSIS_SHA],
    check=True, capture_output=True, text=True,
).stdout.splitlines()
allowed = ('notebooks/', 'tests/', 'scripts/targeted_dropout_colab_helpers.py')
unexpected = [name for name in changed if not name.startswith(allowed)]
if unexpected:
    raise RuntimeError(f'Analysis revision changes producer-sensitive files: {unexpected}')
print({'producer_sha': PRODUCER_SHA, 'analysis_sha': ANALYSIS_SHA, 'notebook': NOTEBOOK_REVISION})


In [ ]:
# PYTHON CELL
# Small, explicit overlay known to work with the repository's Colab scoring path.
from packaging.requirements import Requirement
from packaging.version import Version
import torch as colab_torch

COLAB_PYTHON = (3, 12)
COLAB_TORCH_MIN = Version('2.10.0')
COLAB_TORCH_MAX = Version('2.12.0')

def validate_colab_runtime(torch_module, phase):
    torch_version = Version(torch_module.__version__.split('+', 1)[0])
    if sys.version_info[:2] != COLAB_PYTHON or not (COLAB_TORCH_MIN <= torch_version < COLAB_TORCH_MAX):
        raise RuntimeError(
            'This notebook requires Python 3.12 and PyTorch 2.10.x or 2.11.x. '
            f'Found Python {sys.version_info.major}.{sys.version_info.minor}, PyTorch {torch_module.__version__}. '
            'Select a supported Colab runtime before installing packages.'
        )
    print(f'accepted Colab runtime ({phase}):', f'Python {sys.version_info.major}.{sys.version_info.minor}', f'PyTorch {torch_module.__version__}')
    return torch_version

colab_torch_version = validate_colab_runtime(colab_torch, 'before installation')

pyproject_path = OLMO_DIR / 'pyproject.toml'
pyproject_text = pyproject_path.read_text()
legacy_torch_requirement = '    "torch>=2.1,<2.3",'
colab_torch_requirements = (
    "    \"torch>=2.1,<2.3; python_version < '3.12'\",\n"
    "    \"torch>=2.10,<2.12; python_version >= '3.12'\","
)
if legacy_torch_requirement in pyproject_text:
    pyproject_path.write_text(pyproject_text.replace(legacy_torch_requirement, colab_torch_requirements))
elif not all(requirement in pyproject_text for requirement in colab_torch_requirements.splitlines()):
    raise RuntimeError('Pinned pyproject has an unexpected Torch requirement; update the explicit Colab overlay.')
print('applied verified Colab Torch metadata overlay:', colab_torch_requirements.replace('\n', ' | '))

PIP_OVERLAY = [
    'omegaconf==2.3.0',
    'rich==13.9.4',
    'cached_path==1.8.10',
    'packaging==24.2',
    'boto3==1.35.94',
    'google-cloud-storage==2.19.0',
    'wandb==0.19.1',
    'torchmetrics==1.6.1',
    'datasets==3.2.0',
    'huggingface_hub==0.27.1',
    'transformers==4.47.1',
    'tokenizers==0.21.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PIP_OVERLAY], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(OLMO_DIR)],
    check=True,
)

from importlib.metadata import requires
installed_torch_requirements = [item for item in (requires('ai2-olmo') or []) if Requirement(item).name.lower() == 'torch']
parsed_torch_requirements = [Requirement(item) for item in installed_torch_requirements]
active_torch_requirements = [req for req in parsed_torch_requirements if req.marker is None or req.marker.evaluate()]
expected_torch_specifier = Requirement('torch>=2.10,<2.12').specifier
if not any(req.specifier == expected_torch_specifier for req in active_torch_requirements):
    raise RuntimeError(f'Installed OLMo metadata does not declare Colab Torch support: {installed_torch_requirements}')
print('installed OLMo Torch requirements:', installed_torch_requirements)

sys.path.insert(0, str(OLMO_DIR))
import numpy as np
import pandas as pd
import pyarrow
import torch
from olmo.config import TrainConfig

torch_version = validate_colab_runtime(torch, 'after installation')
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())
print('numpy:', np.__version__, 'pandas:', pd.__version__, 'pyarrow:', pyarrow.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA after installation.')

RUNTIME_IDENTITY = {
    'python': f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}',
    'torch': torch.__version__, 'cuda': torch.version.cuda,
}
print('runtime identity:', RUNTIME_IDENTITY)

In [ ]:
# PYTHON CELL
# Fail before GPU work if the pushed revisions lack any required producer or analysis capability.
import importlib

metrics_source = (OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py').read_text()
score_source = (OLMO_DIR / 'olmo/score.py').read_text()
writer_source = (OLMO_DIR / 'olmo/data/dict_memmap_dataset.py').read_text()
required_markers = {
    'score-index table alignment': 'metadata_and_full_scores_indexed_by_score_index' in metrics_source,
    'bounded scorer index allocation': 'max_entries=' in score_source,
    'shard-continuous stochastic seeds': 'data_start_step or 0' in score_source,
    'bounded writer capacity': 'max_entries: Optional[int]' in writer_source,
}
missing = [name for name, present in required_markers.items() if not present]
if missing:
    raise RuntimeError('Producer revision lacks required capabilities: ' + ', '.join(missing))

helpers = importlib.import_module('scripts.targeted_dropout_colab_helpers')
helpers = importlib.reload(helpers)
required_helpers = [
    'TargetedDropoutRunner', 'TargetedDropoutWorkflow', 'prepare_fixed_subset',
    'build_shard_plan', 'verify_bundle_archive',
]
missing_helpers = [name for name in required_helpers if not hasattr(helpers, name)]
if missing_helpers:
    raise RuntimeError(f'Analysis helper is missing APIs: {missing_helpers}')
for script in ['scripts/21_dropout_uncertainty_metrics.py', 'scripts/22_dropout_strategy_sweep.py']:
    probe = subprocess.run(
        [sys.executable, str(OLMO_DIR / script), '--help'], cwd=str(OLMO_DIR),
        capture_output=True, text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'CLI probe failed for {script}:\n{probe.stderr[-2000:]}')
print('analysis helper:', helpers.__file__)
print('capability probes:', required_markers)


## 3. Configure Paths and Stage Gate

In [ ]:
# PYTHON CELL
import json
from datetime import datetime, timezone

RUN_STAGE = 'stage_c_500k'
ENABLE_STAGE_C = True
STAGE_C_CONFIG_IDS = ['dropout_embed_p0005', 'dropout_resid_p001']

NUM_SAMPLES = 8
SEED = 1
SUBSET_SELECTION_SEED = 1729
SEQ_LEN = 512
GLOBAL_BATCH_SIZE = 32
MICROBATCH = 16
SHARD_ROWS = 24_992
SMOKE_ROWS = 320
BENCH_ROWS = 640
TAU64_CUTOFF = 0.3513622284
FILE_SEQS = 1_048_576

TOKENS_DRIVE = DATA_DRIVE / 'score_pool_tokens_official_500k.npy'
META_DRIVE = DATA_DRIVE / 'score_pool_meta_official_500k.parquet'
FULL_SCORES_DRIVE = SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_full.parquet'
PAIR_MID2_DRIVE = SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_pair_mid2.parquet'
BROAD_DROPOUT_ANALYSIS = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty' / 'dropout_k8_p005' / 'analysis'
PRIOR_CHECKPOINT = MODELS_DRIVE / 'prior'
BOOKS_CHECKPOINT = MODELS_DRIVE / 'conditional_books'

CONFIG_BY_ID = {
    'dropout_attn_p001': {
        'config_id': 'dropout_attn_p001', 'dropout_target': 'attention',
        'attention_dropout': 0.01, 'residual_dropout': 0.0, 'embedding_dropout': 0.0,
        'purpose': 'Stage A best attention-only target',
    },
    'dropout_resid_p001': {
        'config_id': 'dropout_resid_p001', 'dropout_target': 'residual',
        'attention_dropout': 0.0, 'residual_dropout': 0.01, 'embedding_dropout': 0.0,
        'purpose': 'Stage A best residual-or-combined target',
    },
    'dropout_embed_p0005': {
        'config_id': 'dropout_embed_p0005', 'dropout_target': 'embedding',
        'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.005,
        'purpose': 'Stage A best overall rank-preserving target',
    },
}
STAGE_B_IDS = ['dropout_attn_p001', 'dropout_resid_p001', 'dropout_embed_p0005']
STAGE_B_ROOT = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty-targeted' / 'stage_b_100k'
STAGE_B_ACCEPTANCE = STAGE_B_ROOT / 'stage_acceptance.json'

if RUN_STAGE == 'stage_b_100k':
    SUBSET_ID = RUN_STAGE
    STAGE_ROWS = 100_000
    TARGET_CONFIGS = [CONFIG_BY_ID[cid] for cid in STAGE_B_IDS]
elif RUN_STAGE == 'stage_c_500k':
    if not ENABLE_STAGE_C:
        raise RuntimeError('Stage C is locked. Set ENABLE_STAGE_C=True only after reviewing Stage B acceptance.')
    if not STAGE_B_ACCEPTANCE.exists():
        raise FileNotFoundError(f'Stage B acceptance manifest is required: {STAGE_B_ACCEPTANCE}')
    stage_b_gate = json.loads(STAGE_B_ACCEPTANCE.read_text())
    if not stage_b_gate.get('stage_passed'):
        raise RuntimeError('Stage B did not pass; Stage C must not run.')
    if stage_b_gate.get('olmo_sha') != PRODUCER_SHA:
        raise RuntimeError(
            'Stage B and Stage C must use the same pinned producer commit. '
            f"Stage B used {stage_b_gate.get('olmo_sha')}, current pin is {PRODUCER_SHA}."
        )
    allowed = stage_b_gate.get('promoted_config_ids', [])
    chosen = STAGE_C_CONFIG_IDS or stage_b_gate.get('recommended_stage_c_config_ids', [])
    if not 1 <= len(chosen) <= 2:
        raise RuntimeError(f'Stage C requires 1-2 confirmed configs, found: {chosen}')
    if len(set(chosen)) != len(chosen) or any(cid not in allowed for cid in chosen):
        raise RuntimeError(f'Stage C configs must be distinct Stage B promotions. allowed={allowed}, chosen={chosen}')
    SUBSET_ID = RUN_STAGE
    STAGE_ROWS = 500_000
    TARGET_CONFIGS = [CONFIG_BY_ID[cid] for cid in chosen]
else:
    raise ValueError(f'Unsupported RUN_STAGE: {RUN_STAGE}')

STAGE_ROOT = SOURCE_RESULTS_DRIVE / 'dropout-uncertainty-targeted' / SUBSET_ID
RAW_SCORE_DRIVE = STAGE_ROOT / 'raw_score_shards'
CONFIG_DRIVE = DRIVE / 'runtime_configs' / 'dropout-uncertainty-targeted' / SUBSET_ID
SUBSET_MANIFEST = STAGE_ROOT / 'subset_manifest.json'
SOURCE_ROWS_DRIVE = STAGE_ROOT / 'subset_source_rows.npy'
RUN_STATE_PATH = STAGE_ROOT / 'run_state.json'
REPORT_DRIVE = STAGE_ROOT / 'report'
LOCAL_WORK = Path('/content/targeted_dropout_followup') / SUBSET_ID
RUNTIME_CONFIG_DIR = LOCAL_WORK / 'runtime_configs'
RUNTIME_CHECKPOINT_DIR = LOCAL_WORK / 'runtime_checkpoints'
for path in [STAGE_ROOT, RAW_SCORE_DRIVE, CONFIG_DRIVE, REPORT_DRIVE, LOCAL_WORK, RUNTIME_CONFIG_DIR, RUNTIME_CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('stage:', RUN_STAGE, 'rows:', STAGE_ROWS, 'K:', NUM_SAMPLES)
print('configs:', [item['config_id'] for item in TARGET_CONFIGS])
print('stage root:', STAGE_ROOT)

## 4. Validate Inputs and Build the Fixed Subset

In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import SubsetContext, prepare_fixed_subset

PREPARED_SUBSET = prepare_fixed_subset(SubsetContext(
    run_stage=RUN_STAGE, subset_id=SUBSET_ID, stage_rows=STAGE_ROWS,
    seq_len=SEQ_LEN, selection_seed=SUBSET_SELECTION_SEED,
    expected_source_rows=500_000, rows_per_stage_b_pool=20_000,
    tokens_path=TOKENS_DRIVE, metadata_path=META_DRIVE,
    full_scores_path=FULL_SCORES_DRIVE,
    prior_checkpoint=PRIOR_CHECKPOINT, books_checkpoint=BOOKS_CHECKPOINT,
    local_work=LOCAL_WORK, source_rows_path=SOURCE_ROWS_DRIVE,
    subset_manifest_path=SUBSET_MANIFEST,
    producer_sha=PRODUCER_SHA, analysis_sha=ANALYSIS_SHA,
    notebook_revision=NOTEBOOK_REVISION, runtime_identity=RUNTIME_IDENTITY,
))
subset_meta = PREPARED_SUBSET.metadata_path
subset_full = PREPARED_SUBSET.full_scores_path
subset_raw = PREPARED_SUBSET.raw_tokens_path
SUBSET_FINGERPRINT = PREPARED_SUBSET.subset_fingerprint
CHECKPOINT_IDENTITIES = PREPARED_SUBSET.checkpoint_identities
print(SUBSET_MANIFEST.read_text())
print('local raw GB:', round(subset_raw.stat().st_size / 1e9, 3))


In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import build_shard_plan

SHARDS = build_shard_plan(
    STAGE_ROWS, SHARD_ROWS, GLOBAL_BATCH_SIZE, STAGE_ROOT / 'shard_plan.json'
)
print('shards:', len(SHARDS), 'first:', SHARDS[0], 'last:', SHARDS[-1])
print('bounded index bytes per full shard:', SHARD_ROWS * np.dtype(np.int64).itemsize)


## 5. Scoring Helpers

In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import (
    ScoringContext, TargetedDropoutRunner, TargetedDropoutWorkflow, WorkflowContext,
)

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')
SCORING = TargetedDropoutRunner(ScoringContext(
    olmo_dir=OLMO_DIR,
    template_config=OLMO_DIR / 'configs/sweeps/score-targeted-dropout-uncertainty.yaml',
    runtime_checkpoint_dir=RUNTIME_CHECKPOINT_DIR, runtime_config_dir=RUNTIME_CONFIG_DIR,
    config_drive=CONFIG_DRIVE, raw_score_drive=RAW_SCORE_DRIVE,
    stage_root=STAGE_ROOT, subset_raw=subset_raw, run_state_path=RUN_STATE_PATH,
    producer_sha=PRODUCER_SHA, analysis_sha=ANALYSIS_SHA,
    notebook_revision=NOTEBOOK_REVISION, run_stage=RUN_STAGE, subset_id=SUBSET_ID,
    subset_fingerprint=SUBSET_FINGERPRINT, runtime_identity=RUNTIME_IDENTITY,
    checkpoint_identities=CHECKPOINT_IDENTITIES, seed=SEED, num_samples=NUM_SAMPLES,
    global_batch_size=GLOBAL_BATCH_SIZE, stage_rows=STAGE_ROWS,
    shard_rows=SHARD_ROWS, file_seqs=FILE_SEQS,
))

def make_workflow(microbatch):
    return TargetedDropoutWorkflow(WorkflowContext(
        runner=SCORING, olmo_dir=OLMO_DIR, stage_root=STAGE_ROOT,
        config_drive=CONFIG_DRIVE, report_drive=REPORT_DRIVE,
        subset_metadata=subset_meta, subset_full_scores=subset_full,
        subset_manifest=SUBSET_MANIFEST, source_rows_path=SOURCE_ROWS_DRIVE,
        target_configs=TARGET_CONFIGS, shards=SHARDS,
        producer_sha=PRODUCER_SHA, analysis_sha=ANALYSIS_SHA,
        notebook_revision=NOTEBOOK_REVISION, run_stage=RUN_STAGE,
        stage_rows=STAGE_ROWS, num_samples=NUM_SAMPLES, seed=SEED,
        tau64_cutoff=TAU64_CUTOFF, global_batch_size=GLOBAL_BATCH_SIZE,
        shard_rows=SHARD_ROWS, microbatch=int(microbatch),
    ))

build_score_config = SCORING.build_score_config
write_config = SCORING.write_config
run_score_once = SCORING.run_score_once
valid_score_output = SCORING.valid_score_output
load_persisted_microbatch = SCORING.load_persisted_microbatch
shard_output_dir = SCORING.shard_output_dir


In [ ]:
# PYTHON CELL
probe_config = TARGET_CONFIGS[0]
probe_shard = {'start': 0, 'end': GLOBAL_BATCH_SIZE, 'rows': GLOBAL_BATCH_SIZE, 'data_start_step': 0}
probe_cfg = build_score_config(
    probe_config, 'prior_probe', PRIOR_CHECKPOINT, STAGE_ROOT / '_config_probe', probe_shard, MICROBATCH
)
probe_path = write_config(probe_cfg, 'config_probe.yaml')
loaded_probe = TrainConfig.load(str(probe_path), validate_paths=False)
assert loaded_probe.max_duration == 1
assert loaded_probe.data_start_step == 0
assert loaded_probe.uncertainty_scoring.num_samples == NUM_SAMPLES
assert loaded_probe.data.memmap_dtype == 'uint32'
assert loaded_probe.data.paths == [str(subset_raw)]
print('runtime config probe passed:', probe_path)

## 6. Smoke Test / GPU Gate

**Safe to rerun.** This gate is isolated to 320 rows and must pass before production scoring.

In [ ]:
# PYTHON CELL
CONTROL_CONFIG = {
    'config_id': 'dropout_trainmode_p000_smoke', 'dropout_target': 'none',
    'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.0,
    'purpose': 'runtime and row-alignment gate',
}
smoke_result = make_workflow(MICROBATCH).run_zero_dropout_smoke(
    CONTROL_CONFIG, PRIOR_CHECKPOINT, BOOKS_CHECKPOINT, SMOKE_ROWS
)
print('smoke gate passed:', smoke_result)


## 7. Batch-Size and Shard-Size Tuning

**Benchmark only. Safe to rerun.** Each candidate is isolated and bounded to 640 rows (327,680 tokens). Valid candidates with throughput metrics are reused; candidates missing metrics are rebuilt. The fastest successful microbatch is persisted to Drive.

In [ ]:
# PYTHON CELL
benchmark_root = STAGE_ROOT / '_smoke_and_benchmark' / 'benchmark'
benchmark_shard = {'start': 0, 'end': BENCH_ROWS, 'rows': BENCH_ROWS, 'data_start_step': 0}
benchmark_results, selected_result = SCORING.benchmark_microbatches(
    config=CONTROL_CONFIG,
    model_id='prior',
    checkpoint_path=PRIOR_CHECKPOINT,
    benchmark_root=benchmark_root,
    shard=benchmark_shard,
    candidates=[16, 32],
    seq_len=SEQ_LEN,
)

MICROBATCH = int(selected_result['microbatch'])
measured_tps = float(selected_result['tokens_per_second'])
startup_seconds = max(
    0.0, selected_result['measured_compute_seconds'] - selected_result['tokens'] / measured_tps
)
production_jobs = len(TARGET_CONFIGS) * 2 * len(SHARDS)
production_tokens = len(TARGET_CONFIGS) * 2 * STAGE_ROWS * SEQ_LEN
estimated_minutes = (production_tokens / measured_tps + production_jobs * startup_seconds) / 60
run_state = {
    'schema_version': 2, 'created_utc': datetime.now(timezone.utc).isoformat(),
    'stage': RUN_STAGE, 'producer_sha': PRODUCER_SHA, 'subset_fingerprint': SUBSET_FINGERPRINT,
    'analysis_sha': ANALYSIS_SHA, 'notebook_revision': NOTEBOOK_REVISION,
    'num_samples': NUM_SAMPLES, 'microbatch': MICROBATCH, 'shard_rows': SHARD_ROWS,
    'benchmark': selected_result,
}
RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2, sort_keys=True) + '\n')

print('benchmark results:', benchmark_results)
print('selected microbatch:', MICROBATCH)
print('ETA formula: total production tokens / measured tokens/sec + jobs * measured startup overhead')
print('measured tokens/sec:', round(measured_tps, 1), 'startup seconds/job:', round(startup_seconds, 1))
print('conservative full-stage estimate minutes:', round(estimated_minutes, 1))
print('production shards:', len(SHARDS), 'rows per full shard:', SHARD_ROWS)

## 8. Full Resumable Stage Run

**Full run. Safe to rerun.** Valid fingerprint-matched shards are skipped; invalid isolated shards are rebuilt.

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
print('using persisted production microbatch:', MICROBATCH)

for config in TARGET_CONFIGS:
    for model_id, checkpoint in [('prior', PRIOR_CHECKPOINT), ('books', BOOKS_CHECKPOINT)]:
        for shard in SHARDS:
            run_score_once(
                config,
                model_id,
                checkpoint,
                shard_output_dir(config['config_id'], model_id, shard),
                shard,
                MICROBATCH,
            )
print('all expected production shards completed')

## 9. Resume After Disconnect / Status

**Safe to rerun. Fresh-runtime checklist:** rerun Sections 1-5 in order; do not rerun Sections 6-7 when `run_state.json` already exists; run this status cell; rerun Section 8 to repair only invalid shards; then continue with Sections 10-11.

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
WORKFLOW = make_workflow(MICROBATCH)
status = pd.DataFrame(WORKFLOW.status())
print('valid shards:', int(status['valid'].sum()), '/', len(status))
print(status.groupby(['config_id', 'model_id'])['valid'].agg(['sum', 'count']).to_string())
incomplete = status.loc[~status['valid'], ['config_id', 'model_id', 'start', 'end']]
if len(incomplete):
    print('\nIncomplete shards. Rerun Section 8; valid shards will be skipped.')
    print(incomplete.to_string(index=False))
else:
    print('\nAll shards are complete. Continue to Section 10.')
status


## 10. Metrics, Strategy Sweep, and Persisted Outputs

**Safe to rerun.** This cell validates the complete raw grid, removes only an invalid per-config analysis directory, regenerates it, and validates artifact contents before continuing.

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
WORKFLOW = make_workflow(MICROBATCH)
WORKFLOW.analyze()


## 11. Output Review, Promotion Gate, and Download Bundle

In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
WORKFLOW = make_workflow(MICROBATCH)
review = WORKFLOW.build_report()
stage_summary = review['stage_summary']
acceptance = review['acceptance']
print(stage_summary.to_string(index=False))
print('\nacceptance:', json.dumps(acceptance, indent=2, sort_keys=True))


In [ ]:
# PYTHON CELL
MICROBATCH = load_persisted_microbatch()
WORKFLOW = make_workflow(MICROBATCH)
ARCHIVE_NAME = f'targeted_dropout_ladder_{SUBSET_ID}.zip'
ARCHIVE_PATH = Path('/content') / ARCHIVE_NAME
DRIVE_ARCHIVE_PATH = STAGE_ROOT / 'bundles' / ARCHIVE_NAME
bundle = WORKFLOW.build_bundle(ARCHIVE_PATH, DRIVE_ARCHIVE_PATH)
print('verified bundle files:', bundle['file_count'])
print('local bundle:', bundle['archive_path'])
print('Drive fallback:', bundle['drive_archive_path'])
print('size MB:', round(bundle['drive_archive_path'].stat().st_size / 1e6, 2))
print('extract locally to:', bundle['manifest']['local_extract_root'])
print('then run:', bundle['manifest']['local_report_command'])


In [ ]:
# PYTHON CELL
from google.colab import files
from scripts.targeted_dropout_colab_helpers import verify_bundle_archive

AUTO_DOWNLOAD = globals().get('AUTO_DOWNLOAD', True)
archive_name = f'targeted_dropout_ladder_{SUBSET_ID}.zip'
drive_fallback = STAGE_ROOT / 'bundles' / archive_name
local_archive = Path('/content') / archive_name
download_path = drive_fallback if drive_fallback.is_file() else local_archive
verify_bundle_archive(download_path)
if AUTO_DOWNLOAD:
    print('starting browser download:', download_path.name)
    files.download(str(download_path))
else:
    print('automatic download disabled')
print('Drive/manual-download fallback:', drive_fallback)
